##### Task 15
Create a data quality report.

What I want in the end:
- A single report or table showing:
- duplicate customers
- duplicate items
- duplicate orders
- missing customers in orders
- missing items in order arrays
- invalid dates
- null prices
- Each issue type should have a count and affected records

In [0]:
%sql
USE SCHEMA e_commerce;

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW duplicate_customers AS
SELECT customer_id, COUNT(customer_id) count
FROM bronze_customers
GROUP BY customer_id
HAVING COUNT(customer_id) > 1;

CREATE OR REPLACE TEMP VIEW duplicate_items AS
SELECT item_id, COUNT(item_id) count
FROM bronze_items
GROUP BY item_id
HAVING COUNT(item_id) > 1;

CREATE OR REPLACE TEMP VIEW duplicate_orders AS
SELECT order_id, COUNT(order_id) count
FROM bronze_orders
GROUP BY order_id
HAVING COUNT(order_id) > 1;

CREATE OR REPLACE TEMP VIEW missing_customers_from_orders AS
SELECT o.order_id, o.customer_id
FROM bronze_orders o
LEFT JOIN bronze_customers c
  ON o.customer_id = c.customer_id
WHERE c.customer_id IS NULL;

CREATE OR REPLACE TEMP VIEW missing_items_in_orders AS
SELECT o.order_id, o.customer_id, o.item_id
FROM (
  SELECT order_id, customer_id, item.item_id
  FROM bronze_orders
  LATERAL VIEW explode(items) AS item
) o
LEFT JOIN bronze_items i
  ON o.item_id = i.item_id
WHERE i.item_id IS NULL;

CREATE OR REPLACE TEMP VIEW invalid_dates AS
SELECT 
  'orders' table_name, 
  COUNT(o.order_id) total, 
  collect_set(
    named_struct(
      'customer_id', CAST(NULL AS string),
      'order_id', o.order_id,
      'item_id', CAST(NULL AS string)
    )
  ) affected_ids
FROM (
  SELECT order_id, order_date
  FROM bronze_orders
  WHERE try_to_date(order_date, 'yyyy-MM-dd') IS NULL
) o
UNION ALL
SELECT 
  'customers', 
  COUNT(customer_id), 
  collect_set(
    named_struct(
      'customer_id', customer_id,
      'order_id', CAST(NULL AS string),
      'item_id', CAST(NULL AS string)
    )
  ) affected_ids
FROM (
  SELECT customer_id, created_at
  FROM bronze_customers
  WHERE try_to_date(created_at, 'yyyy-MM-dd') IS NULL
);

CREATE OR REPLACE TEMP VIEW null_prices AS
SELECT item_id, price
FROM bronze_items
WHERE price IS NULL OR price <= 0;

CREATE OR REPLACE TEMP VIEW quality_report AS
SELECT 
  'duplicate_customers' report_type, 
  COUNT(customer_id) total, 
  collect_set(
    named_struct(
      'customer_id', customer_id,
      'order_id', CAST(NULL AS string),
      'item_id', CAST(NULL AS string)
    )
  ) affected_ids 
FROM duplicate_customers
UNION ALL
SELECT 
  'duplicate_items', 
  COUNT(item_id), 
  collect_set(
    named_struct(
      'customer_id', CAST(NULL AS string),
      'order_id', CAST(NULL AS string),
      'item_id', item_id
    )
  ) 
FROM duplicate_items
UNION ALL
SELECT 
  'duplicate_orders', 
  COUNT(order_id), 
  collect_set(
    named_struct(
      'customer_id', CAST(NULL AS string),
      'order_id', order_id,
      'item_id', CAST(NULL AS string)
    )
  )
FROM duplicate_orders
UNION ALL
SELECT 
  'missing_customers_in_orders', 
  COUNT(order_id), 
  collect_set(
    named_struct(
      'customer_id', customer_id,
      'order_id', order_id,
      'item_id', CAST(NULL AS string)
    )
  )
FROM missing_customers_from_orders
UNION ALL
SELECT 
  'missing_items_in_orders', 
  COUNT(order_id), 
  collect_set(
    named_struct(
      'customer_id', CAST(NULL AS string),
      'order_id', order_id, 
      'item_id', item_id
    )
  )
FROM missing_items_in_orders
UNION ALL
SELECT 'invalid_dates', SUM(total), flatten(collect_list(affected_ids)) 
FROM invalid_dates
UNION ALL
SELECT 
  'null_prices', 
  COUNT(item_id), 
  collect_list(
    named_struct(
      'customer_id', CAST(NULL AS string),
      'order_id', CAST(NULL AS string),
      'item_id', item_id
    )
  ) 
FROM null_prices;

SELECT *
FROM quality_report;